This file is mostly from a tutorial by geeksforgeeks about fine-tuning a BERT model to work with a sample dataset. Some small modifications have been made for readibility.

[Fine-tuning BERT model for Sentiment Analysis](https://www.geeksforgeeks.org/nlp/fine-tuning-bert-model-for-sentiment-analysis/)

This example uses a dataset that has has analyzes the sentiment of movie reviews


In [4]:
%pip install -q transformers torch scikit-learn pandas

In [5]:
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### Loading the dataset


In [6]:
df = pd.read_csv(
    '/content/IMDB_Dataset.csv',
    engine='python',
    on_bad_lines='skip'
)

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


### Converts the sentiment values to numerical values

For the final project, we need to change how we handle sentiments, as goEmotions is a multiclass dataset


In [7]:
df['sentiment'] = df['sentiment'].map({
    'negative': 0,
    'positive': 1
})

print(df.head())
print(df['sentiment'].value_counts())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
sentiment
1    25000
0    25000
Name: count, dtype: int64


### Creating a smaller dataset


In [8]:
train_text, temp_text, train_labels, temp_labels = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.20,
    random_state=2021,
    stratify=df['sentiment']
)

val_text, test_text, val_labels, test_labels = train_test_split(
    temp_text,
    temp_labels,
    test_size=0.50,
    random_state=2021,
    stratify=temp_labels
)

print("Training samples:", len(train_text))
print("Validation samples:", len(val_text))
print("Test samples:", len(test_text))

Training samples: 40000
Validation samples: 5000
Test samples: 5000


### Loading BERT

We need to modify this to specify that BERT is working on a multi-class problem when it comes down to it


In [9]:
tokenizer = BertTokenizerFast.from_pretrained(
    'bert-base-uncased'
)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
    # problem_type="multi_label_classification",
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
max_length = 64

### Making a dataset class


In [11]:
class IMDBDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length
        )

        encoding['labels'] = self.labels[index]

        return encoding

### Formalizing the datasets


In [12]:
train_dataset = IMDBDataset(
    train_text,
    train_labels,
    tokenizer,
    max_length
)

val_dataset = IMDBDataset(
    val_text,
    val_labels,
    tokenizer,
    max_length
)

test_dataset = IMDBDataset(
    test_text,
    test_labels,
    tokenizer,
    max_length
)

### Tokenizing the data


In [13]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [14]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=data_collator
)

### Prepares cuda if available


In [15]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

model = model.to(device)

print("Using device:", device)

Using device: cuda


### Creates an optimizer for BERT paramaters


In [16]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

### Training the model


In [17]:
epochs = 1

scaler = torch.amp.GradScaler(
    'cuda',
    enabled=(device.type == 'cuda')
)

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_dataloader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == 'cuda')
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)

    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"Training Loss: {avg_loss:.3f}"
    )

Epoch 1/1 - Training Loss: 0.400


### Evaluating the model


In [18]:
model.eval()

val_preds = []
val_labels_list = []

with torch.no_grad():

    for batch in val_dataloader:

        labels = batch['labels']

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        val_preds.extend(
            predictions.cpu().numpy()
        )

        val_labels_list.extend(
            labels.numpy()
        )

print(
    classification_report(
        val_labels_list,
        val_preds,
        target_names=[
            'Negative',
            'Positive'
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.92      0.73      0.81      2500
    Positive       0.78      0.93      0.85      2500

    accuracy                           0.83      5000
   macro avg       0.85      0.83      0.83      5000
weighted avg       0.85      0.83      0.83      5000



In [19]:
model.eval()

test_preds = []
test_labels_list = []

with torch.no_grad():

    for batch in test_dataloader:

        labels = batch['labels']

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        test_preds.extend(
            predictions.cpu().numpy()
        )

        test_labels_list.extend(
            labels.numpy()
        )

In [20]:
print(
    classification_report(
        test_labels_list,
        test_preds,
        target_names=[
            'Negative',
            'Positive'
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.91      0.75      0.82      2500
    Positive       0.79      0.93      0.85      2500

    accuracy                           0.84      5000
   macro avg       0.85      0.84      0.84      5000
weighted avg       0.85      0.84      0.84      5000

